<a href="https://colab.research.google.com/github/j156734119/Amazon-reviews-2023-electronics-SFT-DPO/blob/main/Amazon_Review_Alignment_A100%E2%80%94output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Amazon Review Alignment: A100 Smoke -> Formal -> Baselines

Single Colab workflow: first run an isolated A100 smoke test with Qwen3.5-2B, then run the formal A100 online profile, and finally evaluate trained policies together with the configured baselines.

## 1. Mount Drive And Sync The Repo

Training outputs and checkpoints are stored in Google Drive. Before opening this notebook in Colab, push your latest local changes to GitHub `main`; this notebook clones or fast-forwards the Drive workspace to the latest remote commit.

In [1]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/j156734119/Amazon-reviews-2023-electronics-SFT-DPO.git"
REPO_DIR = Path("/content/drive/MyDrive/amazon-review-alignment-workspace/repo")
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / ".git").exists():
    status = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--short"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if status:
        print("Preserving Colab-local tracked changes before pull:")
        print(status)
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "stash", "push", "-m", "colab-auto-stash-before-pull"],
            check=True,
        )
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

Mounted at /content/drive
Preserving Colab-local tracked changes before pull:
M outputs/a100-qwen3.5-2b/evaluation/evaluation_summary.json
 M outputs/a100-qwen3.5-2b/evaluation/judge_decisions.jsonl
 M outputs/a100-qwen3.5-2b/evaluation/judge_pairwise_summary.csv
 M outputs/a100-qwen3.5-2b/evaluation/metrics.csv
 M outputs/a100-qwen3.5-2b/evaluation/metrics.png
 M outputs/a100-qwen3.5-2b/evaluation/report.md
 M outputs/a100-qwen3.5-2b/rlhf/data_manifest.json
 M outputs/a100-qwen3.5-2b/rlhf/grpo_log_history.json
 M outputs/a100-qwen3.5-2b/rlhf/grpo_metrics.json
 M outputs/a100-qwen3.5-2b/rlhf/ppo_log_history.json
 M outputs/a100-qwen3.5-2b/rlhf/ppo_metrics.json
?? a100_mini_results.zip
Repository: /content/drive/MyDrive/amazon-review-alignment-workspace/repo


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

## 2. Install Dependencies

After this cell finishes, use the Colab menu **Runtime -> Restart session**. After restart, continue from the next cell; do not reinstall unless the runtime was reset.

In [2]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,eval,dev]"],
    cwd=REPO_DIR,
    check=True,
)
print("Installation complete. Restart the Colab runtime now.")

Installation complete. Restart the Colab runtime now.


## 3. Restore Environment And CLI Helper

Add `OPENAI_API_KEY` in Colab Secrets. `HF_TOKEN` is optional but recommended for Hugging Face download limits. `DEEPSEEK_API_KEY` is only required when running the DeepSeek baseline.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata
from packaging.version import Version

drive.mount("/content/drive", force_remount=False)
REPO_DIR = Path("/content/drive/MyDrive/amazon-review-alignment-workspace/repo")
os.chdir(REPO_DIR)

source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

for secret_name in ("OPENAI_API_KEY", "HF_TOKEN", "DEEPSEEK_API_KEY"):
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None
    if value:
        os.environ[secret_name] = value

try:
    import importlib.metadata
    torchao_version = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None:
    print("Removing unused TorchAO:", torchao_version)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)


def training_stack_is_usable() -> bool:
    try:
        import amazon_review_alignment
        import bitsandbytes
        import peft
        import transformers
        import trl
    except (ImportError, ModuleNotFoundError):
        return False
    return Version(bitsandbytes.__version__) >= Version("0.46.1")


if not training_stack_is_usable():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,eval,dev]"],
        cwd=REPO_DIR,
        check=True,
    )

import amazon_review_alignment
import bitsandbytes
import peft
import transformers
import trl


def cli(*arguments: str, check: bool = True) -> subprocess.CompletedProcess:
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(argument) for argument in arguments),
    ]
    print("\n$", " ".join(command), flush=True)
    env = os.environ.copy()
    env["PYTHONPATH"] = source_dir + os.pathsep + env.get("PYTHONPATH", "")
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONFAULTHANDLER"] = "1"
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    output_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        output_lines.append(line)
    returncode = process.wait()
    result = subprocess.CompletedProcess(command, returncode, stdout="".join(output_lines), stderr=None)
    if check and returncode:
        print("\n===== LAST 200 LINES =====")
        print("".join(output_lines[-200:]))
        raise RuntimeError(f"Command failed with exit code {returncode}: " + " ".join(command))
    return result

print("Package:", Path(amazon_review_alignment.__file__).resolve())
print("Training stack:", transformers.__version__, trl.__version__, peft.__version__, bitsandbytes.__version__)
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("HF token loaded:", bool(os.getenv("HF_TOKEN")))
print("DeepSeek key loaded:", bool(os.getenv("DEEPSEEK_API_KEY")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Package: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/src/amazon_review_alignment/__init__.py
Training stack: 5.5.4 1.5.1 0.19.1 0.49.2
OpenAI key loaded: True
HF token loaded: True
DeepSeek key loaded: True


## 4. A100 Environment Check

In [2]:
import importlib.metadata

import torch
import transformers
import trl

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a Colab GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name)
print(f"VRAM: {total_gib:.2f} GiB")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", importlib.metadata.version("peft"))

if "a100" not in gpu_name.lower():
    raise RuntimeError("Expected A100, but Colab assigned: " + gpu_name)
if total_gib < 38:
    raise RuntimeError("Insufficient GPU memory for this A100 profile.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This profile requires BF16 support.")

GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GiB
PyTorch: 2.11.0+cu128
Transformers: 5.5.4
TRL: 1.5.1
PEFT: 0.19.1


## 5. Build The Smoke Config

The smoke config is derived from the formal `rlhf_a100_online_v2.yaml` profile. It still uses Qwen3.5-2B and the A100/BF16 path, but reduces data size, training steps, PPO/GRPO prompt counts, and evaluation samples. Its output directory is isolated from the formal run, so it will not overwrite formal checkpoints.

In [3]:
import json
from pathlib import Path

import yaml

from amazon_review_alignment.config import load_config

FORMAL_CONFIG = "configs/rlhf_a100_online_v2.yaml"
SMOKE_CONFIG_PATH = Path("/content/rlhf_a100_smoke.yaml")

formal = load_config(REPO_DIR / FORMAL_CONFIG)
formal.pop("_config_path", None)

old_root = "outputs/a100-qwen3.5-2b"
smoke_root = "outputs/a100-smoke-qwen3.5-2b"


def replace_output_paths(value):
    if isinstance(value, dict):
        return {key: replace_output_paths(item) for key, item in value.items()}
    if isinstance(value, list):
        return [replace_output_paths(item) for item in value]
    if isinstance(value, str):
        return value.replace(old_root, smoke_root)
    return value


smoke = replace_output_paths(formal)
smoke["project"]["output_dir"] = smoke_root
smoke["data"].update(
    {
        "sample_size": 60,
        "max_scanned_reviews": 20000,
        "rating_targets": {"1": 12, "2": 12, "3": 12, "4": 12, "5": 12},
        "splits": {"train": 42, "validation": 6, "test": 12},
    }
)
smoke["teacher"].update({"pilot_size": 5, "max_estimated_cost_usd": 1.0})
smoke["training"]["sft"]["max_steps"] = 1
smoke["training"]["dpo"]["max_steps"] = 1
smoke["rlhf"].update(
    {
        "human_calibration_samples": 0,
        "ai_reward_train_pairs": 4,
        "ai_reward_validation_pairs": 2,
        "ppo_prompt_count": 4,
    }
)
smoke["rlhf"]["reward"]["max_steps"] = 1
smoke["rlhf"]["ppo"].update({"total_episodes": 4, "gradient_accumulation_steps": 1, "save_steps": 1})
smoke["rlhf"]["grpo"].update({"prompt_count": 4, "num_generations": 2, "generation_batch_size": 2, "max_steps": 1, "save_steps": 1})
smoke["evaluation"].update(
    {
        "max_test_samples": 4,
        "variants": ["base", "sft", "dpo", "ppo", "grpo"],
        "judge_samples_per_pair": 4,
        "judge_pairs": [["sft", "dpo"], ["sft", "ppo"], ["sft", "grpo"], ["ppo", "grpo"]],
    }
)

SMOKE_CONFIG_PATH.write_text(yaml.safe_dump(smoke, sort_keys=False, allow_unicode=True), encoding="utf-8")

print("Smoke config:", SMOKE_CONFIG_PATH)
print("Formal config:", FORMAL_CONFIG)
print("Smoke output:", smoke["project"]["output_dir"])
print("Formal output:", formal["project"]["output_dir"])
print("Formal evaluation variants:", ", ".join(formal["evaluation"]["variants"]))

Smoke config: /content/rlhf_a100_smoke.yaml
Formal config: configs/rlhf_a100_online_v2.yaml
Smoke output: outputs/a100-smoke-qwen3.5-2b
Formal output: outputs/a100-qwen3.5-2b
Formal evaluation variants: base, sft, dpo, ppo, grpo, qwen35_2b_fewshot, nlptown_template, deepseek_v4_pro_fewshot


## 6. A100 Smoke Test

This cell runs the full minimal pipeline in order. If `teacher-batch` is not complete yet, rerun this cell later after the batch finishes. Do not skip smoke and jump directly to the formal run.

In [5]:
import pandas as pd


def output_root_for(config_path: str | Path) -> Path:
    return Path(load_config(config_path)["project"]["output_dir"]).resolve()


def require_teacher_outputs(config_path: str | Path) -> None:
    root = output_root_for(config_path)
    train_preferences = root / "teacher" / "preferences_train.jsonl"
    validation_preferences = root / "teacher" / "preferences_validation.jsonl"
    if not train_preferences.exists() or not validation_preferences.exists():
        raise RuntimeError("Teacher batch is not complete yet. Re-run teacher-batch later, then rerun this cell.")
    print("Teacher train rows:", sum(1 for line in train_preferences.open(encoding="utf-8") if line.strip()))
    print("Teacher validation rows:", sum(1 for line in validation_preferences.open(encoding="utf-8") if line.strip()))


def ensure_teacher_pilot(config_path: str | Path) -> None:
    root = output_root_for(config_path)
    pilot_summary = root / "teacher" / "pilot_summary.json"
    if pilot_summary.exists():
        print("Reusing teacher pilot summary:", pilot_summary)
        return
    cli("teacher-pilot", "--config", str(config_path))


SMOKE_CONFIG = str(SMOKE_CONFIG_PATH)
subprocess.run([sys.executable, "-m", "pytest"], cwd=REPO_DIR, check=True)
cli("prepare-data", "--config", SMOKE_CONFIG)
cli("evaluate", "--config", SMOKE_CONFIG, "--variants", "base", "--force-inference")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets first.")
ensure_teacher_pilot(SMOKE_CONFIG)
cli("teacher-batch", "--config", SMOKE_CONFIG)
require_teacher_outputs(SMOKE_CONFIG)

cli("train-sft", "--config", SMOKE_CONFIG)
cli("merge-sft", "--config", SMOKE_CONFIG)
cli("train-dpo", "--config", SMOKE_CONFIG)
cli("build-rlhf-data", "--config", SMOKE_CONFIG)
cli("train-reward", "--config", SMOKE_CONFIG)
cli("train-ppo", "--config", SMOKE_CONFIG)
cli("train-grpo", "--config", SMOKE_CONFIG)
cli("evaluate", "--config", SMOKE_CONFIG, "--variants", "base", "sft", "dpo", "ppo", "grpo", "--force-inference")
cli("build-report", "--config", SMOKE_CONFIG)

display(pd.read_csv(output_root_for(SMOKE_CONFIG) / "evaluation" / "metrics.csv"))
print("Smoke report:", output_root_for(SMOKE_CONFIG) / "evaluation" / "report.md")


$ /usr/bin/python3 -u -m amazon_review_alignment.cli prepare-data --config /content/rlhf_a100_smoke.yaml
2026-07-16 11:45:20,069 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-07-16 11:45:20,534 | INFO | datasets | TensorFlow version 2.20.0 available.
2026-07-16 11:45:20,536 | INFO | datasets | JAX version 0.7.2 available.
2026-07-16 11:45:20,862 | INFO | amazon_review_alignment.data | Streaming reviews from https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Electronics.jsonl
2026-07-16 11:45:21,606 | INFO | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"
2026-07-16 11:45:21,861 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/McAuley-Lab/Amazon-Reviews-2023/revision/main "HTTP/1.1 200 OK"
2026-07-16 11:45:22,101 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main

,variant,examples,schema_valid_rate,evidence_grounded_rate,word_limit_ok_rate,instruction_following_rate
0,base,4,1.0,1.0,1.0,1.0
1,sft,4,1.0,1.0,1.0,1.0
2,dpo,4,1.0,1.0,1.0,1.0
3,ppo,4,1.0,1.0,1.0,1.0
4,grpo,4,1.0,1.0,1.0,1.0


Smoke report: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-smoke-qwen3.5-2b/evaluation/report.md


## 7. Formal Training: Data And Teacher Preferences

The formal profile uses `configs/rlhf_a100_online_v2.yaml`, including DPO v2, 1,024 shared PPO/GRPO prompts, and baseline evaluation settings. If the batch is still running, you can disconnect the GPU runtime and resume from this cell later.

In [7]:
import json
from pathlib import Path

teacher_dir = FORMAL_ROOT / "teacher"
print("Teacher dir:", teacher_dir)
print("Exists:", teacher_dir.exists())

for name in [
    "pilot_summary.json",
    "batch_state.json",
    "batch_prepare_summary.json",
    "batch_summary.json",
    "batch_quarantine.jsonl",
]:
    path = teacher_dir / name
    print("\n==", name, "==")
    print("exists:", path.exists())
    if path.exists():
        text = path.read_text(encoding="utf-8")
        print(text[:4000])

Teacher dir: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/teacher
Exists: True

== pilot_summary.json ==
exists: True
{
  "requested_items": 100,
  "completed_items": 53,
  "quarantined_items": 47,
  "usage": {
    "input_tokens": 23026,
    "output_tokens": 7023
  },
  "pilot_standard_cost_usd": 0.048873,
  "projected_batch_items": 4000,
  "projected_batch_usage": {
    "input_tokens": 1737811,
    "output_tokens": 530038
  },
  "projected_batch_cost_usd": 1.844265,
  "price_assumptions": {
    "input_price_per_million": 0.75,
    "output_price_per_million": 4.5,
    "batch_discount_multiplier": 0.5
  }
}

== batch_state.json ==
exists: True
{
  "id": "batch_6a2f615a5d4c81908c06a49ef4bae222",
  "completion_window": "24h",
  "created_at": 1781490010,
  "endpoint": "/v1/responses",
  "input_file_id": "file-9wFHWz6wkAqSnkxCZkm5Qu",
  "object": "batch",
  "status": "completed",
  "cancelled_at": null,
  "cancelling_at": null,
  "completed_at": 1781

In [8]:
cli("teacher-batch", "--config", FORMAL_CONFIG, "--verbose", check=False)


$ /usr/bin/python3 -u -m amazon_review_alignment.cli teacher-batch --config configs/rlhf_a100_online_v2.yaml --verbose
2026-07-16 12:02:58,943 | DEBUG | openai._base_client | Request options: {'method': 'get', 'url': '/batches/batch_6a2f615a5d4c81908c06a49ef4bae222', 'security': {'bearer_auth': True}}
2026-07-16 12:02:58,945 | DEBUG | openai._base_client | Sending HTTP Request: GET https://api.openai.com/v1/batches/batch_6a2f615a5d4c81908c06a49ef4bae222
2026-07-16 12:02:58,946 | DEBUG | httpcore.connection | connect_tcp.started host='api.openai.com' port=443 local_address=None timeout=5.0 socket_options=None
2026-07-16 12:02:58,953 | DEBUG | httpcore.connection | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7d8eb75b77d0>
2026-07-16 12:02:58,953 | DEBUG | httpcore.connection | start_tls.started ssl_context=<ssl.SSLContext object at 0x7d8eb8032bd0> server_hostname='api.openai.com' timeout=5.0
2026-07-16 12:02:58,960 | DEBUG | httpcore.connection | st

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'amazon_review_alignment.cli', 'teacher-batch', '--config', 'configs/rlhf_a100_online_v2.yaml', '--verbose'], returncode=1, stdout='2026-07-16 12:02:58,943 | DEBUG | openai._base_client | Request options: {\'method\': \'get\', \'url\': \'/batches/batch_6a2f615a5d4c81908c06a49ef4bae222\', \'security\': {\'bearer_auth\': True}}\n2026-07-16 12:02:58,945 | DEBUG | openai._base_client | Sending HTTP Request: GET https://api.openai.com/v1/batches/batch_6a2f615a5d4c81908c06a49ef4bae222\n2026-07-16 12:02:58,946 | DEBUG | httpcore.connection | connect_tcp.started host=\'api.openai.com\' port=443 local_address=None timeout=5.0 socket_options=None\n2026-07-16 12:02:58,953 | DEBUG | httpcore.connection | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7d8eb75b77d0>\n2026-07-16 12:02:58,953 | DEBUG | httpcore.connection | start_tls.started ssl_context=<ssl.SSLContext object at 0x7d8eb8032bd0> server_hostname=\'

In [9]:
require_teacher_outputs(FORMAL_CONFIG)

Teacher train rows: 1033
Teacher validation rows: 130


In [6]:
FORMAL_ROOT = output_root_for(FORMAL_CONFIG)
print("Formal root:", FORMAL_ROOT)

cli("prepare-data", "--config", FORMAL_CONFIG)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets first.")
ensure_teacher_pilot(FORMAL_CONFIG)
cli("teacher-batch", "--config", FORMAL_CONFIG)
require_teacher_outputs(FORMAL_CONFIG)

Formal root: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b

$ /usr/bin/python3 -u -m amazon_review_alignment.cli prepare-data --config configs/rlhf_a100_online_v2.yaml
2026-07-16 11:54:14,648 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-07-16 11:54:15,122 | INFO | datasets | TensorFlow version 2.20.0 available.
2026-07-16 11:54:15,123 | INFO | datasets | JAX version 0.7.2 available.
2026-07-16 11:54:15,448 | INFO | amazon_review_alignment.data | Streaming reviews from https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Electronics.jsonl
2026-07-16 11:54:16,199 | INFO | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"
2026-07-16 11:54:16,564 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/McAuley-Lab/Amazon-Reviews-2023/revision/main "HTTP/1.1 200 OK"
2026-07-16 11:54:16,806 | INFO | h

RuntimeError: Command failed with exit code 1: /usr/bin/python3 -u -m amazon_review_alignment.cli teacher-batch --config configs/rlhf_a100_online_v2.yaml

In [11]:
from amazon_review_alignment.utils import read_jsonl

root = output_root_for(FORMAL_CONFIG)

train = {row["id"] for row in read_jsonl(root / "data" / "train.jsonl")}
val = {row["id"] for row in read_jsonl(root / "data" / "validation.jsonl")}

pref_train = read_jsonl(root / "teacher" / "preferences_train.jsonl")
pref_val = read_jsonl(root / "teacher" / "preferences_validation.jsonl")

print("train prefs:", len(pref_train))
print("val prefs:", len(pref_val))

print("train ids all in current train split:", all(row["id"] in train for row in pref_train))
print("val ids all in current validation split:", all(row["id"] in val for row in pref_val))

print("train overlap:", sum(row["id"] in train for row in pref_train), "/", len(pref_train))
print("val overlap:", sum(row["id"] in val for row in pref_val), "/", len(pref_val))

print("teacher models:", sorted({row.get("teacher_model") for row in pref_train + pref_val}))

train prefs: 1033
val prefs: 130
train ids all in current train split: True
val ids all in current validation split: True
train overlap: 1033 / 1033
val overlap: 130 / 130
teacher models: ['gpt-5.4-mini-2026-03-17']


In [13]:
root = output_root_for(FORMAL_CONFIG)

paths = {
    "sft": root / "models" / "sft",
    "sft_merged": root / "models" / "sft-merged",
    "dpo": root / "models" / "dpo-v2",
    "ppo": root / "models" / "ppo-v2",
    "grpo": root / "models" / "grpo-v2",
    "test": root / "data" / "test.jsonl",
}

for name, path in paths.items():
    print(name, path, path.exists())

sft /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/models/sft True
sft_merged /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/models/sft-merged True
dpo /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/models/dpo-v2 True
ppo /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/models/ppo-v2 True
grpo /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/models/grpo-v2 True
test /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/data/test.jsonl True


## 8. Formal Training: Stage-By-Stage

Run these cells in order. They are split deliberately so a Colab disconnect or a late-stage failure does not force you to rerun earlier training stages.

### 8.1 SFT Adapter

In [12]:
cli("train-sft", "--config", FORMAL_CONFIG)


$ /usr/bin/python3 -u -m amazon_review_alignment.cli train-sft --config configs/rlhf_a100_online_v2.yaml
2026-07-16 12:08:27,812 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-07-16 12:08:28,279 | INFO | datasets | TensorFlow version 2.20.0 available.
2026-07-16 12:08:28,281 | INFO | datasets | JAX version 0.7.2 available.
2026-07-16 12:08:38,036 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 12:08:38,041 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json "HTTP/1.1 200 OK"
2026-07-16 12:08:38,294 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 12:08:38,300 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c1636

KeyboardInterrupt: 

### 8.2 Merge SFT Adapter

In [ ]:
cli("merge-sft", "--config", FORMAL_CONFIG)

### 8.3 DPO v2 Adapter

In [ ]:
cli("train-dpo", "--config", FORMAL_CONFIG)

### 8.4 Build Reward/PPO/GRPO Data

In [ ]:
cli("build-rlhf-data", "--config", FORMAL_CONFIG)

manifest_path = FORMAL_ROOT / "rlhf" / "data_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))
assert manifest["ppo_grpo_shared_prompt_ids"] is True

### 8.5 Reward Model

In [ ]:
cli("train-reward", "--config", FORMAL_CONFIG)

### 8.6 PPO v2 Adapter

In [ ]:
cli("train-ppo", "--config", FORMAL_CONFIG)

### 8.7 GRPO v2 Adapter

In [ ]:
cli("train-grpo", "--config", FORMAL_CONFIG)

## 9. Evaluate The Five Trained Models

In [14]:
TRAINED_VARIANTS = ["base", "sft", "dpo", "ppo", "grpo"]
cli("evaluate", "--config", FORMAL_CONFIG, "--variants", *TRAINED_VARIANTS, "--force-inference")
cli("build-report", "--config", FORMAL_CONFIG)

display(pd.read_csv(FORMAL_ROOT / "evaluation" / "metrics.csv"))
print("Report:", FORMAL_ROOT / "evaluation" / "report.md")


$ /usr/bin/python3 -u -m amazon_review_alignment.cli evaluate --config configs/rlhf_a100_online_v2.yaml --variants base sft dpo ppo grpo --force-inference
2026-07-16 12:09:50,074 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-07-16 12:09:56,868 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 12:09:56,874 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json "HTTP/1.1 200 OK"
2026-07-16 12:09:57,134 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 12:09:57,140 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-16 12:09:57,375 | INFO | ht

,variant,examples,schema_valid_rate,evidence_grounded_rate,word_limit_ok_rate,instruction_following_rate
0,base,500,0.954,0.842,0.954,0.842
1,sft,500,1.000,0.960,1.000,0.960
2,dpo,500,0.846,0.800,0.846,0.800
3,ppo,500,1.000,0.952,1.000,0.952
4,grpo,500,1.000,0.948,1.000,0.948


Report: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/evaluation/report.md


## 10. Baseline Inference And Unified Evaluation

By default this runs the local Qwen few-shot and nlptown template baselines. If `DEEPSEEK_API_KEY` is configured, it also runs the DeepSeek few-shot baseline.

In [15]:
BASELINE_VARIANTS = [
    "qwen35_2b_fewshot",
    "nlptown_template",
]
if os.getenv("DEEPSEEK_API_KEY"):
    BASELINE_VARIANTS.append("deepseek_v4_pro_fewshot")
else:
    print("Skipping deepseek_v4_pro_fewshot because DEEPSEEK_API_KEY is not set.")
for variant in BASELINE_VARIANTS:
    cli("inference", "--config", FORMAL_CONFIG, "--variant", variant)

ALL_EVAL_VARIANTS = [*TRAINED_VARIANTS, *BASELINE_VARIANTS]
cli("evaluate", "--config", FORMAL_CONFIG, "--variants", *ALL_EVAL_VARIANTS)
cli("build-report", "--config", FORMAL_CONFIG)

metrics = pd.read_csv(FORMAL_ROOT / "evaluation" / "metrics.csv")
display(metrics)
print("Variants evaluated:", ", ".join(ALL_EVAL_VARIANTS))
print("Report:", FORMAL_ROOT / "evaluation" / "report.md")


$ /usr/bin/python3 -u -m amazon_review_alignment.cli inference --config configs/rlhf_a100_online_v2.yaml --variant qwen35_2b_fewshot
2026-07-16 16:55:27,253 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-07-16 16:55:28,065 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 16:55:28,071 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json "HTTP/1.1 200 OK"
2026-07-16 16:55:28,337 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 16:55:28,343 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-16 16:55:28,573 | INFO | httpx | HTTP Request: HE

,variant,examples,schema_valid_rate,evidence_grounded_rate,word_limit_ok_rate,instruction_following_rate
0,base,500,0.954,0.842,0.954,0.842
1,sft,500,1.000,0.960,1.000,0.960
2,dpo,500,0.846,0.800,0.846,0.800
3,ppo,500,1.000,0.952,1.000,0.952
4,grpo,500,1.000,0.948,1.000,0.948
5,qwen35_2b_fewshot,500,0.894,0.794,0.894,0.794
6,nlptown_template,500,1.000,1.000,1.000,1.000
7,deepseek_v4_pro_fewshot,500,0.948,0.938,0.948,0.938


Variants evaluated: base, sft, dpo, ppo, grpo, qwen35_2b_fewshot, nlptown_template, deepseek_v4_pro_fewshot
Report: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/evaluation/report.md


## 11. Optional: AI Blind Judge

This step calls the OpenAI judge API and can incur API cost. It reuses existing prediction files and does not pass `--force-inference`.

In [16]:
RUN_AI_JUDGE = True
AI_JUDGE_SAMPLES_PER_PAIR = 100

if RUN_AI_JUDGE:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets first.")
    cli(
        "evaluate",
        "--config",
        FORMAL_CONFIG,
        "--variants",
        *ALL_EVAL_VARIANTS,
        "--llm-judge",
        "--judge-samples-per-pair",
        str(AI_JUDGE_SAMPLES_PER_PAIR),
    )
    cli("build-report", "--config", FORMAL_CONFIG)
    display(pd.read_csv(FORMAL_ROOT / "evaluation" / "judge_pairwise_summary.csv"))
    print("Report:", FORMAL_ROOT / "evaluation" / "report.md")
else:
    print("Set RUN_AI_JUDGE = True to run blinded pairwise judging.")


$ /usr/bin/python3 -u -m amazon_review_alignment.cli evaluate --config configs/rlhf_a100_online_v2.yaml --variants base sft dpo ppo grpo qwen35_2b_fewshot nlptown_template deepseek_v4_pro_fewshot --llm-judge --judge-samples-per-pair 100
2026-07-16 17:49:25,141 | INFO | numexpr.utils | NumExpr defaulting to 12 threads.
2026-07-16 17:49:26,917 | INFO | amazon_review_alignment.inference | Reusing existing predictions: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/evaluation/predictions/base.jsonl
2026-07-16 17:49:26,953 | INFO | amazon_review_alignment.inference | Reusing existing predictions: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/evaluation/predictions/sft.jsonl
2026-07-16 17:49:26,990 | INFO | amazon_review_alignment.inference | Reusing existing predictions: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/evaluation/predictions/dpo.jsonl
2026-07-16 17:49:27,025 

,comparison,examples,left_model,right_model,right_model_win_rate_ties_half,ci_95_low,ci_95_high,ties
0,base_vs_sft,100,base,sft,0.525,0.430000,0.615,5
1,base_vs_dpo,100,base,dpo,0.575,0.480000,0.670,5
2,base_vs_ppo,100,base,ppo,0.450,0.355000,0.545,4
3,base_vs_grpo,100,base,grpo,0.495,0.400000,0.590,7
4,sft_vs_dpo,100,sft,dpo,0.605,0.515000,0.695,13
5,sft_vs_ppo,100,sft,ppo,0.535,0.460000,0.610,43
6,sft_vs_grpo,100,sft,grpo,0.480,0.400000,0.560,30
7,dpo_vs_ppo,100,dpo,ppo,0.475,0.385000,0.560,21
8,dpo_vs_grpo,100,dpo,grpo,0.410,0.325000,0.495,22
9,ppo_vs_grpo,100,ppo,grpo,0.505,0.440000,0.570,57


Report: /content/drive/MyDrive/amazon-review-alignment-workspace/repo/outputs/a100-qwen3.5-2b/evaluation/report.md
